# Pol-ISARSpaceTarget-1.0 Classification: Clean Complex CNN

This notebook is intentionally classification-only.

It implements exactly:

1. Load full-polarimetric complex ISAR channels using the dataset author's
   `.dat` layout: `HH`, `HV`, `VH`, `VV`.
2. Build a Complex CNN.
3. Extract/tap a feature vector from a fully connected layer.
4. Extract magnitude and phase features.
5. Normalize magnitude and phase features to `[-1, 1]`.
6. Concatenate complex-CNN FC features with magnitude-phase FC features.
7. Feed the fused vector to a classifier.
8. Compute classification metrics.

There is no detection head, no objectness loss, no detection weight, and no box loss.



In [1]:
import json
import math
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
except Exception:
    accuracy_score = confusion_matrix = f1_score = precision_score = recall_score = roc_auc_score = None


print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))




PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Configuration



In [2]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
@dataclass
class Config:
    data_root: str = "/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0"
    output_dir: str = "/content/drive/MyDrive/drdo/outputs/polisar_complex_classification_clean"
    image_size: int = 256
    batch_size: int = 8
    num_workers: int = 0
    epochs: int = 25
    learning_rate: float = 3e-4
    grad_clip_norm: float = 1.0
    early_stop_patience: int = 8
    seed: int = 42
    amp: bool = False
    log_every_batches: int = 10


cfg = Config()
local_candidate = Path.cwd() / "Pol_ISARSpaceTarget1.0"
if not Path(cfg.data_root).exists() and local_candidate.exists():
    cfg.data_root = str(local_candidate)
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2))


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(cfg.seed)




{
  "data_root": "/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0",
  "output_dir": "/content/drive/MyDrive/drdo/outputs/polisar_complex_classification_clean",
  "image_size": 256,
  "batch_size": 8,
  "num_workers": 0,
  "epochs": 25,
  "learning_rate": 0.0003,
  "grad_clip_norm": 1.0,
  "early_stop_patience": 8,
  "seed": 42,
  "amp": false,
  "log_every_batches": 10
}


## Dataset Loader

The model receives only the complex tensor. The filename is used only to create
the class label.

The reader below follows `Tools/DataReadPython.py` from the dataset:
each `.dat` file stores interleaved float32 real and imaginary values, reshaped
to `256 x 256`, and the four channels are placed as the `2 x 2` polarimetric
matrix `[HH HV; VH VV]`.



In [4]:
POLS = ("HH", "HV", "VH", "VV")
SATELLITE_CLASSES = ("Capella", "HISEA1", "QPSSAR", "Radarsat2", "StarLink", "Worldview4")
CLASS_TO_ID = {name: idx for idx, name in enumerate(SATELLITE_CLASSES)}
DUPLICATE_RE = re.compile(r" \(\d+\)(?=\.[^.]+$)")


def is_drive_duplicate_name(name: str) -> bool:
    return bool(DUPLICATE_RE.search(name))


def infer_satellite_id(base: str) -> int:
    for class_name, class_id in CLASS_TO_ID.items():
        if base.startswith(class_name + "_"):
            return class_id
    raise ValueError(f"Could not infer class from filename base: {base}")


def read_full_pol_complex_official_layout(image_dir: Path, base: str) -> np.ndarray:
    data_size = (256, 256, 2, 2)
    complex_data = np.zeros(data_size, dtype=np.complex64)
    for i in range(2):
        for j in range(2):
            idx = i * 2 + j
            file_path = image_dir / f"{base}_{POLS[idx]}.dat"
            combined_data = np.fromfile(file_path, dtype=np.float32)
            if combined_data.size != 256 * 256 * 2:
                raise ValueError(f"Unexpected file size for {file_path}: {combined_data.size} float32 values")
            real_part = combined_data[0::2].reshape(data_size[0:2])
            imag_part = combined_data[1::2].reshape(data_size[0:2])
            complex_data[:, :, i, j] = real_part + 1j * imag_part
    return complex_data


def polarimetric_matrix_to_channels(complex_data: np.ndarray) -> np.ndarray:
    return np.stack(
        [
            complex_data[:, :, 0, 0],  # HH
            complex_data[:, :, 0, 1],  # HV
            complex_data[:, :, 1, 0],  # VH
            complex_data[:, :, 1, 1],  # VV
        ],
        axis=0,
    ).astype(np.complex64)


def resize_complex_tensor(x: torch.Tensor, image_size: int) -> torch.Tensor:
    if x.shape[-2:] == (image_size, image_size):
        return x
    real = F.interpolate(x.real.unsqueeze(0), size=(image_size, image_size), mode="bilinear", align_corners=False)
    imag = F.interpolate(x.imag.unsqueeze(0), size=(image_size, image_size), mode="bilinear", align_corners=False)
    return torch.complex(real.squeeze(0), imag.squeeze(0))


def normalize_complex_per_channel(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    max_amp = torch.abs(x).amax(dim=(-2, -1), keepdim=True).clamp_min(eps)
    return x / max_amp.to(x.dtype)


class PolISARClassificationDataset(Dataset):
    def __init__(self, data_root: str, split: str, image_size: int) -> None:
        self.data_root = Path(data_root)
        self.split = split
        self.image_size = image_size
        self.image_dir = self.data_root / "ImageSets" / "images" / split
        if not self.image_dir.exists():
            raise FileNotFoundError(f"Missing image directory: {self.image_dir}")

        bases = sorted(
            {
                p.name.rsplit("_", 1)[0]
                for p in self.image_dir.glob("*_HH.dat")
                if not is_drive_duplicate_name(p.name)
            }
        )
        self.samples = [base for base in bases if all((self.image_dir / f"{base}_{pol}.dat").exists() for pol in POLS)]
        expected = {"train": 1314, "valid": 162, "test": 162}.get(split)
        print(f"{split}: usable complete full-pol samples = {len(self.samples)}; expected = {expected}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        base = self.samples[index]
        complex_matrix = read_full_pol_complex_official_layout(self.image_dir, base)
        x = torch.from_numpy(polarimetric_matrix_to_channels(complex_matrix))
        x = resize_complex_tensor(x, self.image_size)
        x = normalize_complex_per_channel(x)
        return {"x": x, "class_id": torch.tensor(infer_satellite_id(base), dtype=torch.long), "base": base}


def collate_batch(batch: Sequence[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    return {
        "x": torch.stack([item["x"] for item in batch]),
        "class_id": torch.stack([item["class_id"] for item in batch]),
        "base": [str(item["base"]) for item in batch],
    }


train_ds = PolISARClassificationDataset(cfg.data_root, "train", cfg.image_size)
valid_ds = PolISARClassificationDataset(cfg.data_root, "valid", cfg.image_size)
test_ds = PolISARClassificationDataset(cfg.data_root, "test", cfg.image_size)
print("Dataset sizes:", len(train_ds), len(valid_ds), len(test_ds))
sample = train_ds[0]
print("Sample:", sample["base"], sample["x"].shape, sample["x"].dtype, sample["class_id"])


def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_batch,
    )


train_loader = make_loader(train_ds, shuffle=True)
valid_loader = make_loader(valid_ds, shuffle=False)
test_loader = make_loader(test_ds, shuffle=False)




train: usable complete full-pol samples = 1314; expected = 1314
valid: usable complete full-pol samples = 162; expected = 162
test: usable complete full-pol samples = 162; expected = 162
Dataset sizes: 1314 162 162
Sample: Capella_el_40_az_0 torch.Size([4, 256, 256]) torch.complex64 tensor(0)


## Magnitude and Phase Features

Magnitude and phase are explicitly separated and normalized to `[-1, 1]`.



In [5]:
def magnitude_phase_tensor(x: torch.Tensor, eps: float = 1e-6) -> Tuple[torch.Tensor, torch.Tensor]:
    mag = torch.abs(x)
    mag_max = mag.amax(dim=(-2, -1), keepdim=True).clamp_min(eps)
    mag_norm = 2.0 * (mag / mag_max) - 1.0
    phase_norm = torch.angle(x) / math.pi
    return mag_norm.float(), phase_norm.float()


mag_check, phase_check = magnitude_phase_tensor(sample["x"].unsqueeze(0))
print("Magnitude range:", float(mag_check.min()), float(mag_check.max()))
print("Phase range:", float(phase_check.min()), float(phase_check.max()))




Magnitude range: -0.9999931454658508 1.0
Phase range: -0.9999741315841675 0.9999952912330627


## Complex CNN Architecture

This is the core architecture Mam asked to see.

Complex convolution:

`(a + jb)(c + jd) = (ac - bd) + j(ad + bc)`

The FC feature vector is tapped from `self.fc_complex`.



In [6]:
class ComplexConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, padding: int = 0) -> None:
        super().__init__()
        self.real = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False)
        self.imag = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        real = self.real(x.real) - self.imag(x.imag)
        imag = self.real(x.imag) + self.imag(x.real)
        return torch.complex(real, imag)


class ComplexBatchNorm2d(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.real_bn = nn.BatchNorm2d(channels)
        self.imag_bn = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.complex(self.real_bn(x.real), self.imag_bn(x.imag))


class ComplexModReLU(nn.Module):
    def __init__(self, channels: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mag = torch.abs(x)
        activated = F.relu(mag + self.bias)
        phase = x / (mag + self.eps)
        return activated.to(x.dtype) * phase


class ComplexResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = ComplexConv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = ComplexBatchNorm2d(out_channels)
        self.act1 = ComplexModReLU(out_channels)
        self.conv2 = ComplexConv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = ComplexBatchNorm2d(out_channels)
        self.act2 = ComplexModReLU(out_channels)
        self.proj = None
        if stride != 1 or in_channels != out_channels:
            self.proj = nn.Sequential(
                ComplexConv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                ComplexBatchNorm2d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x if self.proj is None else self.proj(x)
        out = self.act1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act2(out + identity)


class MagnitudePhaseFCBranch(nn.Module):
    def __init__(self, out_dim: int = 128) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, out_dim),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mag_norm, phase_norm = magnitude_phase_tensor(x)
        return self.encoder(torch.cat([mag_norm, phase_norm], dim=1))


class ComplexCNNClassifier(nn.Module):
    def __init__(self, num_classes: int = 6) -> None:
        super().__init__()
        self.stem = nn.Sequential(
            ComplexConv2d(4, 32, kernel_size=5, stride=2, padding=2),
            ComplexBatchNorm2d(32),
            ComplexModReLU(32),
        )
        self.stage1 = nn.Sequential(ComplexResidualBlock(32, 48, stride=2), ComplexResidualBlock(48, 48))
        self.stage2 = nn.Sequential(ComplexResidualBlock(48, 96, stride=2), ComplexResidualBlock(96, 96))
        self.stage3 = nn.Sequential(ComplexResidualBlock(96, 160, stride=2), ComplexResidualBlock(160, 160))

        # FC layer where complex CNN features are tapped.
        self.fc_complex = nn.Sequential(
            nn.Linear(320, 192),
            nn.GELU(),
            nn.Linear(192, 128),
            nn.Tanh(),
        )
        self.fc_mag_phase = MagnitudePhaseFCBranch(out_dim=128)

        # Concatenated vector = [complex_fc_features, magnitude_phase_features].
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        z = self.stem(x)
        z = self.stage1(z)
        z = self.stage2(z)
        z = self.stage3(z)

        real_imag_map = torch.cat([z.real, z.imag], dim=1)
        pooled = real_imag_map.mean(dim=(-2, -1))
        complex_fc_features = self.fc_complex(pooled)
        mag_phase_features = self.fc_mag_phase(x)
        fused_features = torch.cat([complex_fc_features, mag_phase_features], dim=1)
        logits = self.classifier(fused_features)
        return {
            "logits": logits,
            "complex_fc_features": complex_fc_features,
            "mag_phase_features": mag_phase_features,
            "fused_features": fused_features,
        }


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ComplexCNNClassifier(num_classes=len(SATELLITE_CLASSES)).to(device)
print(model)
print("Trainable parameters:", f"{count_parameters(model):,}")

with torch.no_grad():
    smoke = sample["x"].unsqueeze(0).to(device)
    out = model(smoke)
    print("logits:", out["logits"].shape)
    print("complex_fc_features:", out["complex_fc_features"].shape, float(out["complex_fc_features"].min()), float(out["complex_fc_features"].max()))
    print("mag_phase_features:", out["mag_phase_features"].shape, float(out["mag_phase_features"].min()), float(out["mag_phase_features"].max()))
    print("fused_features:", out["fused_features"].shape)




ComplexCNNClassifier(
  (stem): Sequential(
    (0): ComplexConv2d(
      (real): Conv2d(4, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), bias=False)
      (imag): Conv2d(4, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), bias=False)
    )
    (1): ComplexBatchNorm2d(
      (real_bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (imag_bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (2): ComplexModReLU()
  )
  (stage1): Sequential(
    (0): ComplexResidualBlock(
      (conv1): ComplexConv2d(
        (real): Conv2d(32, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (imag): Conv2d(32, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      )
      (bn1): ComplexBatchNorm2d(
        (real_bn): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (imag_bn): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True,

## Metrics



In [7]:
def classification_metrics(y_true: Sequence[int], logits: torch.Tensor) -> Dict[str, float]:
    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    pred = probs.argmax(axis=1)
    y = np.array(y_true)
    metrics = {"accuracy": float((pred == y).mean())}
    if accuracy_score is not None:
        metrics.update(
            {
                "precision_macro": float(precision_score(y, pred, average="macro", zero_division=0)),
                "recall_macro": float(recall_score(y, pred, average="macro", zero_division=0)),
                "f1_macro": float(f1_score(y, pred, average="macro", zero_division=0)),
            }
        )
        try:
            metrics["auc_ovr"] = float(roc_auc_score(y, probs, multi_class="ovr", average="macro"))
        except Exception:
            metrics["auc_ovr"] = float("nan")
    return metrics


def move_batch(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {"x": batch["x"].to(device), "class_id": batch["class_id"].to(device), "base": batch["base"]}


def run_epoch(loader: DataLoader, optimizer: torch.optim.Optimizer | None, phase: str) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    logits_all: List[torch.Tensor] = []
    targets: List[int] = []
    correct = 0
    seen = 0

    for batch_idx, batch in enumerate(loader, start=1):
        batch = move_batch(batch, device)
        with torch.set_grad_enabled(is_train):
            outputs = model(batch["x"])
            loss = criterion(outputs["logits"], batch["class_id"])
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                optimizer.step()

        total_loss += float(loss.detach().cpu())
        preds = outputs["logits"].detach().argmax(dim=1)
        correct += int((preds == batch["class_id"]).sum().item())
        seen += int(batch["class_id"].numel())
        logits_all.append(outputs["logits"].detach().cpu())
        targets.extend(batch["class_id"].detach().cpu().tolist())

        if batch_idx == 1 or batch_idx == len(loader) or batch_idx % cfg.log_every_batches == 0:
            print(f"{phase} [{batch_idx:04d}/{len(loader):04d}] loss={total_loss / batch_idx:.4f} acc={correct / max(seen, 1):.3f}", flush=True)

    logits_cat = torch.cat(logits_all, dim=0)
    metrics = classification_metrics(targets, logits_cat)
    metrics["loss"] = total_loss / max(len(loader), 1)
    return metrics




## Training



In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
best_metric = -1.0
epochs_without_improvement = 0
history_rows: List[Dict[str, float]] = []
out_dir = Path(cfg.output_dir)
best_path = out_dir / "best_complex_cnn_classifier.pt"
last_path = out_dir / "last_complex_cnn_classifier.pt"

for epoch in range(cfg.epochs):
    t0 = time.time()
    train_metrics = run_epoch(train_loader, optimizer, phase=f"epoch {epoch + 1} train")
    valid_metrics = run_epoch(valid_loader, None, phase=f"epoch {epoch + 1} valid")

    row = {
        "epoch": float(epoch + 1),
        "lr": float(optimizer.param_groups[0]["lr"]),
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"valid_{k}": v for k, v in valid_metrics.items()},
        "seconds": time.time() - t0,
    }
    history_rows.append(row)
    print(json.dumps(row, indent=2), flush=True)

    current = valid_metrics.get("f1_macro", valid_metrics["accuracy"])
    torch.save({"model": model.state_dict(), "epoch": epoch, "classes": SATELLITE_CLASSES}, last_path)
    if current > best_metric:
        best_metric = current
        epochs_without_improvement = 0
        torch.save({"model": model.state_dict(), "epoch": epoch, "classes": SATELLITE_CLASSES, "valid": valid_metrics}, best_path)
        print("Saved best checkpoint:", best_path)
    else:
        epochs_without_improvement += 1
        print(f"No validation improvement for {epochs_without_improvement}/{cfg.early_stop_patience} epochs.")

    if epochs_without_improvement >= cfg.early_stop_patience:
        print("Early stopping triggered.")
        break

(out_dir / "history.json").write_text(json.dumps(history_rows, indent=2))
print("Training history saved:", out_dir / "history.json")




epoch 1 train [0001/0165] loss=1.8064 acc=0.125
epoch 1 train [0010/0165] loss=1.7913 acc=0.113
epoch 1 train [0020/0165] loss=1.7636 acc=0.212
epoch 1 train [0030/0165] loss=1.7281 acc=0.287
epoch 1 train [0040/0165] loss=1.6776 acc=0.350
epoch 1 train [0050/0165] loss=1.6196 acc=0.385
epoch 1 train [0060/0165] loss=1.5549 acc=0.398
epoch 1 train [0070/0165] loss=1.4881 acc=0.421
epoch 1 train [0080/0165] loss=1.4260 acc=0.441
epoch 1 train [0090/0165] loss=1.3702 acc=0.472
epoch 1 train [0100/0165] loss=1.3073 acc=0.500
epoch 1 train [0110/0165] loss=1.2500 acc=0.528
epoch 1 train [0120/0165] loss=1.1901 acc=0.555
epoch 1 train [0130/0165] loss=1.1331 acc=0.583
epoch 1 train [0140/0165] loss=1.0853 acc=0.603
epoch 1 train [0150/0165] loss=1.0452 acc=0.621
epoch 1 train [0160/0165] loss=1.0085 acc=0.637
epoch 1 train [0165/0165] loss=1.0016 acc=0.639
epoch 1 valid [0001/0021] loss=0.1736 acc=1.000
epoch 1 valid [0010/0021] loss=0.7048 acc=0.700
epoch 1 valid [0020/0021] loss=0.8481 ac

## Test Evaluation



In [9]:
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint["model"])
test_metrics = run_epoch(test_loader, None, phase="test")
print("Test metrics:")
print(json.dumps(test_metrics, indent=2))

@torch.no_grad()
def collect_predictions(loader: DataLoader) -> Dict[str, object]:
    model.eval()
    targets: List[int] = []
    preds: List[int] = []
    bases: List[str] = []
    for batch in loader:
        batch = move_batch(batch, device)
        logits = model(batch["x"])["logits"]
        targets.extend(batch["class_id"].detach().cpu().tolist())
        preds.extend(logits.argmax(dim=1).detach().cpu().tolist())
        bases.extend(batch["base"])
    return {"targets": targets, "predictions": preds, "bases": bases, "classes": list(SATELLITE_CLASSES), "metrics": test_metrics}


predictions = collect_predictions(test_loader)
(out_dir / "test_predictions.json").write_text(json.dumps(predictions, indent=2))
if confusion_matrix is not None:
    cm = confusion_matrix(predictions["targets"], predictions["predictions"])
    np.save(out_dir / "test_confusion_matrix.npy", cm)
    print("Confusion matrix rows=true cols=pred:")
    print(cm)




test [0001/0021] loss=0.0281 acc=1.000
test [0010/0021] loss=0.0104 acc=1.000
test [0020/0021] loss=0.0314 acc=1.000
test [0021/0021] loss=0.0311 acc=1.000
Test metrics:
{
  "accuracy": 1.0,
  "precision_macro": 1.0,
  "recall_macro": 1.0,
  "f1_macro": 1.0,
  "auc_ovr": 1.0,
  "loss": 0.03112633139278928
}
Confusion matrix rows=true cols=pred:
[[27  0  0  0  0  0]
 [ 0 27  0  0  0  0]
 [ 0  0 27  0  0  0]
 [ 0  0  0 27  0  0]
 [ 0  0  0  0 27  0]
 [ 0  0  0  0  0 27]]
